Data source
-----------
Deportation Data Project (DDP) — FOIA-obtained ICE records

URL: https://deportationdata.org/data.html

Downloads: Arrests CSV + Detainers CSV

Coverage: October 2022 – early March 2026

NOTE on aggregation:
The raw DDP files contain ~700K individual arrest records but no county
FIPS identifiers, only state names (arrests) and facility state (detainers).
We aggregate to state level before any further analysis.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path(__file__).resolve().parents[2] / "shared"))

import pandas as pd
from config import DATA_RAW, DATA_INTERIM, STATE_ABBR_TO_FIPS


def load_and_map(fname: str, count_col_hint: str) -> pd.DataFrame:
    path = DATA_RAW / fname
    if not path.exists():
        raise FileNotFoundError(
            f"Missing: {path}\n"
            "Download from https://deportationdata.org/data.html\n"
            "or run notebooks/00_preprocess_raw_ddp.ipynb"
        )
    df = pd.read_csv(path, low_memory=False)
    print(f"  Loaded {fname}: {len(df):,} rows  |  columns: {list(df.columns)}")

    # Find state column (abbr)
    state_col = next(
        (c for c in df.columns if "state" in c.lower()),
        df.columns[0]
    )
    # Find count column
    count_col = next(
        (c for c in df.columns if count_col_hint in c.lower()),
        df.columns[1]
    )
    df["state_fips"] = (
        df[state_col].astype(str).str.strip().str.upper()
        .map(STATE_ABBR_TO_FIPS)
    )
    df = df[df["state_fips"].notna()][["state_fips", count_col]].rename(
        columns={count_col: count_col_hint}
    )
    return df


def main():
    print("Loading ICE state-aggregated enforcement data...")
    arr = load_and_map("ice_arrests.csv",   "arrests")
    det = load_and_map("ice_detainers.csv", "detainers")

    ice = arr.merge(det, on="state_fips", how="outer").fillna(0)
    ice["arrests"]   = ice["arrests"].astype(int)
    ice["detainers"] = ice["detainers"].astype(int)

    out = DATA_INTERIM / "ice_state_outcomes.csv"
    ice.to_csv(out, index=False)
    print(f"\nSaved → {out}  ({len(ice)} states)")
    print(ice.sort_values("arrests", ascending=False).head(10).to_string(index=False))


if __name__ == "__main__":
    main()